# Линейные модели с нуля: сравнение с sklearn

Ноутбук проверяет собственные `LinearRegression` и `LogisticRegression`
(пакет `linear_models`) на реальных и синтетических данных. Каждая модель
обучается и проверяется на **тех же данных**, что и её аналог из `sklearn`.

Разделы:
1. Данные
2. Линейная регрессия: `analytic` и `gradient` против `LinearRegression` и `Ridge`
3. Логистическая регрессия против `sklearn.linear_model.LogisticRegression`
4. Режимы: без штрафа, L1, L2, с `intercept` и без
5. Сходимость: `loss_history_` и `learning_rate`
6. Выводы и объяснение расхождений

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.linear_model import Ridge as SkRidge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from linear_models.linear_regression import LinearRegression
from linear_models.logistic_regression import LogisticRegression

SEED = 42
np.set_printoptions(precision=6, suppress=True)

print("python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("свои классы :", LinearRegression.__name__, LogisticRegression.__name__)
print("sklearn     :", SkLinearRegression.__name__, SkRidge.__name__, SkLogisticRegression.__name__)

## 1. Данные

Четыре набора: для регрессии синтетика с известными весами и `load_diabetes`,
для классификации синтетика и `load_breast_cancer`.

- Одно разбиение train/test на набор: `test_size=0.25`, `random_state=SEED`
  (для классификации со `stratify=y`, чтобы доли классов совпадали).
- Признаки стандартизируются: **среднее и стандартное отклонение считаются
  только по train**, к test применяется то же преобразование.
- Наши модели и модели `sklearn` получают **одни и те же** массивы.
- Масштабирование нужно из-за градиентного спуска с фиксированным
  `learning_rate`: на сырых признаках `load_breast_cancer` он не успевает
  сойтись (это показано в разделе про сходимость).

In [ ]:
TEST_SIZE = 0.25


def split_and_scale(X, y, stratify=None):
    """Одно разбиение train/test и стандартизация по train."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=stratify
    )
    scaler = StandardScaler().fit(X_train)
    return {
        "X_train": scaler.transform(X_train),
        "X_test": scaler.transform(X_test),
        "y_train": y_train,
        "y_test": y_test,
    }


rng = np.random.default_rng(SEED)
n, d = 300, 3
W_TRUE = np.array([2.0, -1.0, 0.5])
B_TRUE = 1.0
X_syn = rng.standard_normal((n, d))
z_syn = X_syn @ W_TRUE + B_TRUE

y_syn_reg = z_syn + 0.5 * rng.standard_normal(n)
y_syn_clf = (rng.random(n) < 1 / (1 + np.exp(-z_syn))).astype(int)

X_dia, y_dia = load_diabetes(return_X_y=True)
X_bc, y_bc = load_breast_cancer(return_X_y=True)

REG_DATA = {
    "synthetic": split_and_scale(X_syn, y_syn_reg),
    "diabetes": split_and_scale(X_dia, y_dia),
}
CLF_DATA = {
    "synthetic": split_and_scale(X_syn, y_syn_clf, stratify=y_syn_clf),
    "breast_cancer": split_and_scale(X_bc, y_bc, stratify=y_bc),
}

for group in (REG_DATA, CLF_DATA):
    for name, s in group.items():
        assert s["X_train"].shape[0] == s["y_train"].shape[0]
        assert s["X_test"].shape[0] == s["y_test"].shape[0]
        assert np.allclose(s["X_train"].mean(axis=0), 0.0, atol=1e-10)
        assert np.allclose(s["X_train"].std(axis=0), 1.0)

for title, group in (("регрессия", REG_DATA), ("классификация", CLF_DATA)):
    print(title)
    for name, s in group.items():
        print(
            f"  {name:14s} train {s['X_train'].shape}, test {s['X_test'].shape}",
            f"| y_train mean {s['y_train'].mean():.4f}, y_test mean {s['y_test'].mean():.4f}",
        )
print("W_TRUE =", W_TRUE, " B_TRUE =", B_TRUE)
print("classes in synthetic y:", np.bincount(y_syn_clf))
print("synthetic X_train mean:", CLF_DATA["synthetic"]["X_train"].mean(axis=0))